# **Lab:** Security in Computer Networks

**Student name:** write your name here

### **Topics**
- Confidentiality
- Public Key Cryptography (RSA)
- Digital Signatures

In [1]:
# Install cryptography library (only first time)
# This library provides tools for encryption, decryption, signatures, etc.
%pip install cryptography

  Using cached cffi-2.0.0-cp313-cp313-macosx_10_13_x86_64.whl.metadata (2.6 kB)
  Using cached pycparser-3.0-py3-none-any.whl.metadata (8.2 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 8.9 MB/s  0:00:01m0:00:0100:01
Using cached cffi-2.0.0-cp313-cp313-macosx_10_13_x86_64.whl (185 kB)
Using cached pycparser-3.0-py3-none-any.whl (48 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [cryptography] [cryptography]
Note: you may need to restart the kernel to use updated packages.


In [2]:
# Import RSA module for asymmetric cryptography
from cryptography.hazmat.primitives.asymmetric import rsa

# Import serialization tools (used for saving/loading keys, not used deeply here)
from cryptography.hazmat.primitives import serialization

# Import padding schemes (needed for secure encryption)
from cryptography.hazmat.primitives.asymmetric import padding

# Import hashing algorithms (SHA256, etc.)
from cryptography.hazmat.primitives import hashes

## **🔐 Part 1:** RSA (Public Key Cryptography)

- Public key → shared
- Private key → secret
- Encrypt with public key, decrypt with private key


In [3]:
# Generate PRIVATE key
# public_exponent=65537 is a standard safe value used in RSA
# key_size=2048 means 2048-bit key (secure for most purposes)
private_key = rsa.generate_private_key(
    public_exponent=65537,   # mathematical parameter used in RSA
    key_size=2048            # size of the key in bits (larger = more secure but slower)
)

# Generate PUBLIC key from private key
# In RSA, both keys are mathematically linked
public_key = private_key.public_key()

print("Keys generated successfully")

Keys generated successfully


In [5]:
# Export public key to PEM
pem_pub = public_key.public_bytes(
    encoding=serialization.Encoding.PEM,
    format=serialization.PublicFormat.SubjectPublicKeyInfo
).decode()

pem_pub

'-----BEGIN PUBLIC KEY-----\nMIIBIjANBgkqhkiG9w0BAQEFAAOCAQ8AMIIBCgKCAQEAmCN0RiO1xdAmioBAdUe7\nmHjSuTYlelRTaJoRj61JRb92TMxQTeUP5EWXIwn3aymym+pFSJnepxMehIyV3RRw\nt3ViJJcyEFDgicVXFLu3IMKiOCNHy0+PbgymUppB3wtKLb/U0saGDqw9i4iq3Ypn\nlLp5a1on2K5E6zxqd7fACMeEN2yMvHenqs39Mpu+RnAYTPGPvIg5UBFSu6bFcZsS\n+Ti8O6iHX51Xxkb2PfW1bgyUixH3LNkC8VjSXpB++DlbzGzJeyua/KwzaLr686QH\n5v+9kLJ+lnuwxU884Jww/ULGMjauKe5ukT1zUj4zpGsFoFFDM3TrbU7SKR+fa8wq\nCQIDAQAB\n-----END PUBLIC KEY-----\n'

In [6]:
# Convert received PEM to an object
received_public_key = serialization.load_pem_public_key(pem_pub.encode())

received_public_key

In [7]:
import base64

# Define message to encrypt
# .encode() converts string → bytes (required for crypto functions)
message = "Hello secure world".encode()

# Encrypt using PUBLIC key
encrypted = received_public_key.encrypt(
    message,  # data to encrypt (must be bytes)
    padding.OAEP(  # OAEP = Optimal Asymmetric Encryption Padding (secure padding)
        mgf=padding.MGF1(algorithm=hashes.SHA256()),  # MGF1 = Mask Generation Function
        algorithm=hashes.SHA256(),  # hash algorithm used internally
        label=None  # optional label (not needed here)
    )
)

print("Encrypted (Python bytes object):", encrypted) # This will show as a bytes object (not human-readable). ASCIIs are not suitable for binary data, so we will convert it to hex and base64 for display.
print("Encrypted (hex):", encrypted.hex())
print("Encrypted (base64):", base64.b64encode(encrypted).decode()) # Standard way to represent binary data as text

Encrypted (Python bytes object): b'\n\xff\x16voKI\xc7\xa4\x1c\xb6xd{q\xc4Q\x93!e]_\xc9\xe5\xc3\xaa\x1f\x0b\x13\xfe\xfc\xaf*\x16U_b+\x97\xa9\xefA\xa8\x18\x10\xd2\xeaA\xdd\x82\xda\x88\x84U \tR\x84\x0c,\xa6\x94[X0\x08>\xaf\xb4\x10]\x16\xf3j\xff\x960\xbfy\x88\x05^\xd1h:{\xf8f}K\xac\xb3oU}\xea=S\\\x96jT_a\xfe\xbe\xd0\x82]\xf8\xdf\xd1\xc0\xdd\x88\xcc\x14\xd5\x0e\x99\xfb\x14\xd0H\x82vw\x8d\xa1\xf3\xa7\x8c\xd5\xdb\xf9\x071\x8875Q#\xfb\x01\xe0\xea2\t9\xa8g$)\x8c\x1b\xca}\x80\xa58Q\x0c\xcb\xdb\x1b\x93n\x0e \x1b\xc2li\xfc\x84\x8a\xb1\xbd\xdb\xec#\x10\xc2\xb7\xf5\x80[\x9a$x\x9b\xc6\x12\xc9\xec\xea[.\x0cJa\x8d\x939~\x11\x1f\xc2O&\xa1:\x9a\xd7\x97\xfe\xb8\xf7S\\\x08\xd8\xc12\x04\x81\x02\x96\xe3\x86\xd9\xc9\xa2\xc7\xf6E\xfa\xe1\x13\xab#\xe7\xee\xfc\xb1ODj\xd5\xfd\xfb\xe1\xe5\x12\xce\x82'
Encrypted (hex): 0aff16766f4b49c7a41cb678647b71c4519321655d5fc9e5c3aa1f0b13fefcaf2a16555f622b97a9ef41a81810d2ea41dd82da888455200952840c2ca6945b5830083eafb4105d16f36aff9630bf7988055ed1683a7bf8667d4bacb36f557dea3d535c9

In [8]:
# Decrypt using PRIVATE key
decrypted = private_key.decrypt(
    encrypted,  # encrypted data
    padding.OAEP(
        mgf=padding.MGF1(algorithm=hashes.SHA256()),
        algorithm=hashes.SHA256(),
        label=None
    )
)

# Convert bytes → string using .decode()
print("Decrypted:", decrypted.decode())

Decrypted: Hello secure world


When you:

1. Take a message  
2. Encrypt it with the **receiver’s public key**  
3. Encode it in **Base64**

You already achieved **confidentiality**:  
Only the holder of the **private key** can decrypt it.

> ⚠️ **Base64 is NOT security**, it is just an encoding format to represent binary data as text.

The real problem (without certificates)

The key question is:

> ❗ How do you know that the **public key** actually belongs to the right person?

Without certificates, this can happen:

- An attacker gives you **their public key**
- You encrypt the message using that key
- The attacker decrypts it with their private key
- Then re-encrypts it using the real recipient’s key

*Result:* your communication was intercepted without you noticing.

This is known as a **Man-in-the-Middle** attack.

So… what are certificates for?

A certificate (e.g., `X.509`) is used to:

**Bind:**
- A **public key**
- To a **real identity** (person, server, organization)

**It also includes:**
- A trusted signature (from a Certificate Authority)
- Owner information
- Expiration/validity dates

Who guarantees this?

A **Certificate Authority (CA)** such as:

- Let's Encrypt  
- DigiCert  

These entities sign certificates and essentially say:

> “Yes, this public key belongs to this entity.”

## **✍️ Part 2:** Digital Signatures

- Sign with PRIVATE key
- Verify with PUBLIC key


In [16]:
# Create digital signature
signature = private_key.sign(
    message,  # original message
    padding.PSS(  # Probabilistic Signature Scheme (secure signature padding)
        mgf=padding.MGF1(hashes.SHA256()),  # mask generation function
        salt_length=padding.PSS.MAX_LENGTH  # random salt for security
    ),
    hashes.SHA256()  # hash algorithm used before signing
)

print("Signature created")

Signature created


In [17]:
# Verify signature
try:
    public_key.verify(
        signature,  # signature to verify
        message,    # original message
        padding.PSS(
            mgf=padding.MGF1(hashes.SHA256()),
            salt_length=padding.PSS.MAX_LENGTH
        ),
        hashes.SHA256()
    )
    print("Signature VALID")
except:
    print("Signature INVALID")

Signature VALID


How it fits together?

In protocols like **Transport Layer Security (TLS)**:

1. The server sends its certificate  
2. The client verifies:
   - The CA signature  
   - The domain name  
3. **Only then** the public key is trusted  
4. The key is used for encryption

## **🌐 Part 3:** TLS Handshake Simulation

Real TLS:
1. Client gets server public key
2. Client creates session key
3. Client encrypts session key with public key
4. Server decrypts
5. Both use symmetric encryption


In [18]:
# Import symmetric encryption tool
from cryptography.fernet import Fernet

In [19]:
# Client generates symmetric session key
# This key will be used for fast encryption (like AES)
session_key = Fernet.generate_key()

# Create cipher object for encryption/decryption
cipher_client = Fernet(session_key)

print("Session key created")

Session key created


In [20]:
# Client encrypts session key using server PUBLIC key
encrypted_session_key = public_key.encrypt(
    session_key,
    padding.OAEP(
        mgf=padding.MGF1(hashes.SHA256()),
        algorithm=hashes.SHA256(),
        label=None
    )
)

In [21]:
# Server decrypts session key using PRIVATE key
decrypted_session_key = private_key.decrypt(
    encrypted_session_key,
    padding.OAEP(
        mgf=padding.MGF1(hashes.SHA256()),
        algorithm=hashes.SHA256(),
        label=None
    )
)

# Server creates cipher with shared key
cipher_server = Fernet(decrypted_session_key)


In [22]:
# Secure communication
secure_message = cipher_client.encrypt(b"Hello TLS secure world")

print("Encrypted message:", secure_message)

print("Decrypted message:", cipher_server.decrypt(secure_message).decode())

Encrypted message: b'gAAAAABp565zB0hTUrUNtIgq4jh8iK6TECgPBk3TUJiqY56jiJAz2H3EwKi9H8cW3UGhbQY5UFo7SsdjBskuETPyp8J_qZOB7MCSQeSembfxPPdNbzesG5c='
Decrypted message: Hello TLS secure world


## **Assignment:** Secure Communication Challenge

### **Objective**
Apply all concepts from the lab:
- Confidentiality (encryption)
- Authentication (digital signatures)
- Integrity (verification)

### **Group Dynamics (Balanced Work)**

Each student must complete ALL roles:

1. **Sender**
2. **Receiver**

You will interact with at least **one other student**.

### **Task 1:** Secure Message Sending

Each student must:

1. Generate an RSA key pair
2. Share ONLY the **public key**
3. Create a message
4. Encrypt the message using the receiver’s public key
5. Sign the message using your private key

👉 Send to another student:
- Encrypted message
- Signature
- Your public key

### **Task 2:** Secure Message Receiving

Each student must:

1. Receive:
   - Encrypted message
   - Signature
   - Sender's public key

2. Decrypt the message using your private key  
3. Verify the signature using the sender’s public key  

✔ If valid → accept message  
❌ If invalid → reject message  

### **Time Limit**
- 20–25 minutes total

### **Evaluation (Checklist)**

| **Criteria** | **Score (10pts)** |
|--------|------|
| RSA encryption implemented | 1 pts |
| Digital signature implemented | 1 pts |
| Correctly decrypted & verified message | 4 pts |
| Explanation reflection questions | 4 pts |

### **Reflection Questions**

1. Why is encryption alone NOT enough for security?  
2. What is the role of digital signatures?  
3. Why does TLS use a hybrid approach?  